# Task 2 — Basic scRNA-seq Tutorial

**Reference:** [Scanpy Basic scRNA-seq Tutorial](https://github.com/scverse/scanpy-tutorials/blob/main/basic-scrna-tutorial.ipynb)

**Dataset:** PBMC 3K (pre-processed in Task 1)  
**Tools:** Scanpy, Leiden algorithm, UMAP

---

## Background

In Task 1, we cleaned and pre-processed the raw data. Now we perform the core biological analysis: grouping similar cells together (**clustering**), visualizing them in 2D (**UMAP**), identifying the genes that define each cluster (**marker genes**), and assigning biological identities to each cluster (**cell type annotation**). This is the standard scRNA-seq analysis workflow used in published research.

---

## Step 0 — Install and Import Libraries

In addition to **Scanpy**, we now need **leidenalg** — a Python package that implements the Leiden community detection algorithm used for clustering cells. The Leiden algorithm is an improvement over the older Louvain algorithm and is now the standard for scRNA-seq clustering. We also import general-purpose libraries **NumPy**, **Pandas**, and **Matplotlib** for data manipulation and visualization.

In [ ]:
# Uncomment the line below if running in Google Colab
# !pip install scanpy leidenalg matplotlib seaborn

import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, facecolor='white', figsize=(7, 5))

print('Scanpy version:', sc.__version__)

## Step 1 — Load Pre-processed Data from Task 1

We load the `.h5ad` file that was saved at the end of Task 1. This file already contains the filtered, normalized, log-transformed, HVG-selected, scaled, and PCA-reduced data — so we can skip directly to the clustering steps. If the Task 1 output file is not found (e.g., running this notebook independently in Colab), we fall back to loading Scanpy's built-in `pbmc3k_processed()` dataset which is already fully pre-processed and includes clustering results.

In [ ]:
import os

# Try to load the file produced by Task 1
if os.path.exists('../01-preprocessing/pbmc3k_preprocessed.h5ad'):
    adata = sc.read_h5ad('../01-preprocessing/pbmc3k_preprocessed.h5ad')
    print('Loaded pre-processed data from Task 1 output.')
else:
    # Fallback: use Scanpy's built-in processed version
    print('Task 1 file not found. Loading Scanpy built-in processed PBMC 3K...')
    adata = sc.datasets.pbmc3k_processed()
    print('Loaded processed PBMC 3K from Scanpy datasets.')

print('\nAnnData object:')
print(adata)

## Step 2 — Compute the Neighborhood Graph

The **k-nearest neighbor (kNN) graph** is the foundation for both UMAP visualization and Leiden clustering. For each cell, we find its `k` most similar cells (neighbors) based on their distance in PCA space, and draw edges connecting them. Cells that are transcriptionally similar (express similar genes) will be close to each other and densely connected in this graph, while cells from different types will be far apart. We use 40 principal components (`n_pcs=40`) as this captures most of the biologically meaningful variation, and `n_neighbors=10` which is a standard choice for PBMC data. The resulting graph is stored in `adata.obsp['connectivities']`.

In [ ]:
# Build kNN graph: connect each cell to its 10 nearest neighbors in 40-PC space
# n_neighbors=10 controls local vs global structure; n_pcs=40 uses top 40 PCs
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)

print('k-nearest neighbor graph computed.')
print('Stored in adata.obsp["connectivities"] and adata.obsp["distances"].')

## Step 3 — Compute UMAP Embedding

**UMAP (Uniform Manifold Approximation and Projection)** is a non-linear dimensionality reduction method that projects the high-dimensional kNN graph into just 2 dimensions for visualization. Unlike PCA which is a linear method, UMAP can capture complex, non-linear relationships in the data. The result is a 2D scatter plot where cells that are transcriptionally similar cluster together, and cells from different types appear as separate 'islands'. UMAP is computed from the kNN graph we built in Step 2, not directly from the expression matrix. The 2D coordinates are stored in `adata.obsm['X_umap']`.

In [ ]:
# Compute UMAP: projects cells into 2D based on the kNN graph
# This step may take 1-2 minutes on the full dataset
sc.tl.umap(adata)

print('UMAP embedding computed.')
print(f'2D coordinates stored in adata.obsm["X_umap"]: shape = {adata.obsm["X_umap"].shape}')
print('Each row is one cell; the two columns are the X and Y UMAP coordinates.')

## Step 4 — Leiden Clustering

**Leiden clustering** is a community detection algorithm applied to the kNN graph. It identifies groups of cells (communities) that are more densely connected to each other than to the rest of the graph — these represent transcriptionally similar cell populations. The **resolution parameter** controls the granularity of clustering: a lower value (e.g., 0.3) gives fewer, larger clusters, while a higher value (e.g., 1.0) gives more, smaller clusters. We use `resolution=0.5` which typically gives a good balance for PBMC data (~8-10 clusters). The cluster assignment for each cell is stored in `adata.obs['leiden']`.

In [ ]:
# Run Leiden clustering on the kNN graph
# resolution=0.5 gives ~8-10 clusters for PBMC data (adjust if needed)
sc.tl.leiden(adata, resolution=0.5)

n_clusters = adata.obs['leiden'].nunique()
print(f'Leiden clustering found {n_clusters} clusters.')
print('\nNumber of cells per cluster:')
print(adata.obs['leiden'].value_counts().sort_index())

## Step 5 — Visualize UMAP Colored by Cluster

Now we plot the UMAP with cells colored by their Leiden cluster assignment. Each color represents a distinct cell population. Cells within each 'island' of the UMAP should all belong to the same cluster — if the clustering worked well, cluster colors should align with the visible groupings in the UMAP. The `legend_loc='on data'` argument places cluster number labels directly on the UMAP rather than in a separate legend, making it easier to read.

In [ ]:
# Plot UMAP with cells colored by Leiden cluster
# Each color = one cluster; labels appear directly on the clusters
sc.pl.umap(
    adata,
    color=['leiden'],
    legend_loc='on data',
    title='PBMC 3K — Leiden Clusters (resolution=0.5)',
    frameon=False,
    save='_leiden_clusters.png'
)
print('UMAP cluster plot saved.')

## Step 6 — Find Marker Genes per Cluster

To understand what each cluster represents biologically, we need to identify **marker genes** — genes that are significantly more highly expressed in one cluster compared to all other clusters. We use the **Wilcoxon rank-sum test**, which is a non-parametric statistical test that compares the expression distribution of each gene between one cluster and the rest. For each cluster, genes are ranked by their statistical significance and fold-change. The results are stored in `adata.uns['rank_genes_groups']` and can be visualized to show the top marker genes for each cluster.

In [ ]:
# Find marker genes: for each cluster, test which genes are significantly
# more expressed than in all other clusters combined (one-vs-rest)
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')

# Plot the top 10 marker genes for each cluster
# Genes shown left-to-right are ranked by statistical significance
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False, save='_rank_genes.png')
print('Marker gene ranking complete and plotted.')

In [ ]:
# Display top 5 marker genes per cluster as a readable table
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names

top_markers = pd.DataFrame(
    {group: result['names'][group][:5] for group in groups}
)
print('Top 5 marker genes per Leiden cluster:')
print(top_markers.to_string())

## Step 7 — Visualize Known Cell-Type Marker Genes on UMAP

We now overlay the expression of **known PBMC marker genes** on the UMAP. These are well-established genes from the biological literature that are specifically expressed in particular immune cell types — for example, `IL7R` and `CCR7` are specific to CD4 T cells, `MS4A1` marks B cells, and `CD14` marks monocytes. By checking whether these known markers are highly expressed in specific UMAP clusters, we can start to identify what cell type each cluster represents. Bright yellow spots on the UMAP indicate cells with high expression of that gene.

In [ ]:
# Well-known PBMC marker genes from the literature
# These are used to match clusters to known cell types
marker_genes = [
    'IL7R',   # CD4 T cells — T cell receptor signaling
    'CCR7',   # Naive CD4 T cells — homing receptor
    'CD14',   # CD14+ Monocytes — innate immune cell
    'LYZ',    # Monocytes — lysozyme, antimicrobial
    'MS4A1',  # B cells — B cell surface marker (CD20)
    'CD8A',   # CD8 T cells — cytotoxic T cell marker
    'GNLY',   # NK cells — granulysin, cytotoxicity
    'NKG7',   # NK cells — natural killer cell granule protein
    'FCER1A', # Dendritic cells — IgE receptor
    'PPBP',   # Megakaryocytes / Platelets
]

# Keep only genes that exist in our filtered dataset
marker_genes = [g for g in marker_genes if g in adata.var_names]
print(f'Plotting {len(marker_genes)} marker genes found in this dataset.')

# Plot each marker gene on the UMAP — color scale = expression level
sc.pl.umap(
    adata,
    color=marker_genes,
    ncols=3,
    frameon=False,
    save='_marker_genes_umap.png'
)
print('Marker gene UMAP plots saved.')

## Step 8 — Dot Plot of Marker Genes per Cluster

A **dot plot** is one of the most informative visualization styles in scRNA-seq analysis. For each combination of (cluster, gene), it shows two things simultaneously: the **size of the dot** represents the fraction of cells in that cluster that express the gene (larger = more cells expressing it), and the **color intensity** represents the average expression level in those cells (darker = higher expression). This makes it easy to see at a glance which genes are specific to which clusters, and how strongly they are expressed. A good marker gene should have a large, dark dot in one cluster and small, light dots in all others.

In [ ]:
# Dot plot: dot size = fraction of cells expressing the gene
#           dot color = mean expression level in expressing cells
sc.pl.dotplot(
    adata,
    marker_genes,
    groupby='leiden',
    save='_dotplot.png'
)
print('Dot plot saved. Use this to match clusters to cell types.')

## Step 9 — Violin Plot of Marker Gene Expression

**Violin plots** provide another view of marker gene expression across clusters. For each gene, we plot its expression distribution within every Leiden cluster. A gene that is a strong marker for a specific cluster will have a wide, high violin in that cluster and flat, near-zero violins in all other clusters. This helps confirm that the marker genes identified in Step 6 are truly specific to their respective clusters and not broadly expressed. Wider violins indicate more cells with that expression level.

In [ ]:
# Violin plot: expression distribution of top 4 marker genes across all clusters
# Wide violin = many cells with that expression; narrow/flat = few cells
sc.pl.violin(
    adata,
    marker_genes[:4],
    groupby='leiden',
    save='_violin_markers.png'
)
print('Violin plot saved.')

## Step 10 — Cell Type Annotation

**Cell type annotation** is the final and most biologically meaningful step. Based on the marker genes we identified and the known PBMC markers from literature, we manually assign a cell type label to each cluster. For example, if cluster 0 shows high expression of `IL7R` and `CCR7`, we label it 'CD4 T cells'. If cluster 3 shows high `MS4A1`, we label it 'B cells'. This mapping is stored as a new column `'cell_type'` in `adata.obs`. In practice, this step requires biological knowledge and literature review — the mapping below is based on the standard PBMC marker gene reference.

In [ ]:
# Manual cell type annotation based on marker gene expression patterns
# Cluster IDs are assigned by Leiden — adjust this mapping based on YOUR results
cell_type_map = {
    '0': 'CD4 T cells',         # High IL7R, CCR7
    '1': 'CD14 Monocytes',      # High CD14, LYZ
    '2': 'CD4 T cells',         # High IL7R
    '3': 'B cells',             # High MS4A1
    '4': 'CD8 T cells',         # High CD8A
    '5': 'NK cells',            # High GNLY, NKG7
    '6': 'CD14 Monocytes',      # High LYZ
    '7': 'Dendritic cells',     # High FCER1A, CST3
    '8': 'Megakaryocytes',      # High PPBP
}

# Add cell type labels to the AnnData object
adata.obs['cell_type'] = adata.obs['leiden'].map(cell_type_map).fillna('Unknown')

print('Cell types assigned:')
print(adata.obs['cell_type'].value_counts())

# Final UMAP colored by cell type — this is the main result of the analysis
sc.pl.umap(
    adata,
    color='cell_type',
    legend_loc='right margin',
    title='PBMC 3K — Annotated Cell Types',
    frameon=False,
    save='_cell_types.png'
)
print('Annotated UMAP saved.')

## Step 11 — Save the Final Annotated Data

We save the fully analyzed AnnData — now containing cluster assignments, UMAP coordinates, marker gene rankings, and cell type annotations — to a new `.h5ad` file. This file represents the complete result of the scRNA-seq analysis workflow and can be shared with collaborators, submitted alongside a publication, or loaded in Task 3 to explore the AnnData structure in detail.

In [ ]:
# Save the fully annotated AnnData to disk
adata.write_h5ad('pbmc3k_clustered.h5ad')

print('Annotated AnnData saved to: pbmc3k_clustered.h5ad')
print('\nFinal AnnData object contains:')
print(adata)
print('\nCell metadata columns:', list(adata.obs.columns))
print('Embeddings:', list(adata.obsm.keys()))

---
## Summary — Basic scRNA-seq Analysis Pipeline

| Step | Method | Key Parameter | Result |
|---|---|---|---|
| kNN graph | Euclidean distance in PCA space | k=10, 40 PCs | Cell connectivity map |
| UMAP | Manifold learning | Default | 2D visualization |
| Leiden clustering | Community detection | resolution=0.5 | ~9 clusters |
| Marker genes | Wilcoxon rank-sum test | one-vs-rest | Top genes per cluster |
| Dot plot | Mean expression + fraction | — | Gene specificity overview |
| Cell type annotation | Manual (literature markers) | — | 7 PBMC cell types |

> **Next step:** Open `03_anndata_tutorial.ipynb` to explore the AnnData data structure that stores all of this data.